In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio

In [ ]:
DIR_DATA = Path("data")
DIR_ICEYE_ALIGNED = DIR_DATA / "1-5_intermediate" / "1_ICEYE_aligned"
DIR_ICEYE_ZSCORE = DIR_DATA / "1-5_intermediate" / "2_ICEYE_Zscore"
FILEPATH_ICEYE_LIST = DIR_DATA / "0_metadata" / "iceye-list-full.csv"

GROUP_OPTIONS = ["year", "orbit_direction", "look_side"]

In [ ]:
def read_sar(path):
    with rasterio.open(path) as src:
        img = src.read(1).astype(np.float32)
    return img

def compute_group_statistics(group_df):
    profile = None
    mean = None

    n = len(group_df)
    # ----------------------------
    # First pass: Mean
    # ----------------------------
    for _, row in group_df.iterrows():
        path = (
            DIR_ICEYE_ALIGNED
            / row["filename"]
        )
        path = DIR_ICEYE_ALIGNED / (
            path.stem + "_EPSG2958_res05m.tif"
        )
        with rasterio.open(path) as src:
            img = src.read(1).astype(np.float32)
            if profile is None:
                profile = src.profile.copy()
            if mean is None:
                mean = np.zeros_like(
                    img,
                    dtype=np.float64
                )
            mean += img
    mean /= n

    # ----------------------------
    # Second pass: Variance
    # ----------------------------
    var = np.zeros_like(
        mean,
        dtype=np.float64
    )
    for _, row in group_df.iterrows():
        path = (
            DIR_ICEYE_ALIGNED
            / row["filename"]
        )
        path = DIR_ICEYE_ALIGNED / (
            path.stem + "_EPSG2958_res05m.tif"
        )
        img = read_sar(path)
        var += (img - mean) ** 2
    var /= n
    std = np.sqrt(var)
    std[std < 1e-6] = 1e-6
    return (
        mean.astype(np.float32),
        std.astype(np.float32),
        profile
    )


def save_geotiff(path, image, profile):
    profile = profile.copy()
    profile.update(
        dtype="float32",
        count=1,
        compress="lzw"
    )
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(
            image.astype(np.float32),
            1
        )

In [ ]:
# --------------------------------------------------
# Read metadata
# --------------------------------------------------
df = pd.read_csv(FILEPATH_ICEYE_LIST)

df["date"] = pd.to_datetime(df["date"])
df["year"] = df["date"].dt.year

df["group"] = (
    df["year"].astype(str)
    + "-"
    + df["orbit_direction"]
    + "-"
    + df["look_side"]
)

# --------------------------------------------------
# Compute statistics
# --------------------------------------------------
for group, group_df in df.groupby("group"):
    print(
        f"{group}: {len(group_df)} images"
    )
    mean, std, profile = compute_group_statistics(
        group_df
    )
    save_geotiff(
        DIR_ICEYE_ZSCORE / f"{group}_mean.tif",
        mean,
        profile
    )
    save_geotiff(
        DIR_ICEYE_ZSCORE / f"{group}_std.tif",
        std,
        profile
    )

print("Done.")